In [5]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# model
from sklearn.linear_model import LogisticRegression

# metrics
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

# save model
import joblib




In [6]:
df = pd.read_csv("/Users/sheetalbohra/Documents/Assignment2.csv")

df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [7]:
# Replace " ?" with NaN
df.replace(" ?", np.nan, inplace=True)

# Drop missing values
df.dropna(inplace=True)

df.shape


(32561, 15)

In [8]:
X = df.drop("income", axis=1)
y = df["income"]


In [9]:
y = y.map({"<=50K": 0, ">50K": 1})


In [10]:
#Identify categorical and numerical columns

categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns
print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)


Categorical: Index(['workclass', 'education', 'marital.status', 'occupation',
       'relationship', 'race', 'sex', 'native.country'],
      dtype='object')
Numerical: Index(['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss',
       'hours.per.week'],
      dtype='object')


In [11]:
#Create preprocessing pipeline
#Logistic Regression requires:
#OneHotEncoding for categorical
#Scaling for numeric


preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


In [12]:
#Create Logistic Regression model pipeline

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)


In [13]:
#Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [14]:
#Train the model

model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [15]:
#Make predictions
y_pred = model.predict(X_test)

# probability required for AUC
y_proba = model.predict_proba(X_test)[:, 1]


In [16]:
#Compute assignment metrics
accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print("Accuracy:", accuracy)
print("AUC:", auc)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("MCC:", mcc)


Accuracy: 0.8542914171656687
AUC: 0.9043452466932171
Precision: 0.7393658159319412
Recall: 0.6096938775510204
F1 Score: 0.6682977979727368
MCC: 0.5804376795941794


In [17]:
#Confusion matrix and report

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))



Confusion Matrix:
[[4608  337]
 [ 612  956]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.93      0.91      4945
           1       0.74      0.61      0.67      1568

    accuracy                           0.85      6513
   macro avg       0.81      0.77      0.79      6513
weighted avg       0.85      0.85      0.85      6513



In [18]:
#Save model for Streamlit app
joblib.dump(model, "logistic_regression.joblib")

print("Model saved successfully")


Model saved successfully


In [19]:
metrics_df = pd.DataFrame({
    "Model": ["Logistic Regression"],
    "Accuracy": [accuracy],
    "AUC": [auc],
    "Precision": [precision],
    "Recall": [recall],
    "F1": [f1],
    "MCC": [mcc]
})

metrics_df.to_csv("logistic_regression_metrics.csv", index=False)

metrics_df


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.854291,0.904345,0.739366,0.609694,0.668298,0.580438


In [20]:
##########  2. Decision Tree classifier #######################

In [21]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)
import pandas as pd
import numpy as np
import joblib


In [22]:
from sklearn.model_selection import train_test_split

X = df.drop("income", axis=1)
y = df["income"].map({"<=50K": 0, ">50K": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [23]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


In [24]:
dt_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_tree),
        ("classifier", DecisionTreeClassifier(
            random_state=42,
            max_depth=10,          # keeps it from overfitting too hard
            min_samples_split=10,
            min_samples_leaf=5
        ))
    ]
)

In [25]:
dt_model.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
y_pred_dt = dt_model.predict(X_test)

# AUC needs probabilities
y_proba_dt = dt_model.predict_proba(X_test)[:, 1]

In [27]:
accuracy_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_proba_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
mcc_dt = matthews_corrcoef(y_test, y_pred_dt)

print("Decision Tree Results")
print("Accuracy:", accuracy_dt)
print("AUC:", auc_dt)
print("Precision:", precision_dt)
print("Recall:", recall_dt)
print("F1:", f1_dt)
print("MCC:", mcc_dt)


Decision Tree Results
Accuracy: 0.8544449562413634
AUC: 0.8921589009719157
Precision: 0.7686308492201039
Recall: 0.5656887755102041
F1: 0.6517266715650257
MCC: 0.572957532876549


In [28]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))


Confusion Matrix:
[[4678  267]
 [ 681  887]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91      4945
           1       0.77      0.57      0.65      1568

    accuracy                           0.85      6513
   macro avg       0.82      0.76      0.78      6513
weighted avg       0.85      0.85      0.85      6513



In [29]:
joblib.dump(dt_model, "decision_tree.joblib")
print("Decision Tree model saved successfully")

Decision Tree model saved successfully


In [30]:
metrics_dt = pd.DataFrame({
    "Model": ["Decision Tree"],
    "Accuracy": [accuracy_dt],
    "AUC": [auc_dt],
    "Precision": [precision_dt],
    "Recall": [recall_dt],
    "F1": [f1_dt],
    "MCC": [mcc_dt]
})

metrics_dt.to_csv("decision_tree_metrics.csv", index=False)
metrics_dt

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Decision Tree,0.854445,0.892159,0.768631,0.565689,0.651727,0.572958


In [31]:
import os
os.listdir()

['.zshrc.save',
 '.config',
 'metrics',
 'Music',
 '.zprofile.pysave',
 '.DS_Store',
 'Housing.ipynb',
 '.CFUserTextEncoding',
 'decision_tree_metrics.csv',
 '.xonshrc',
 'Untitled.ipynb',
 '.zshrc',
 '.local',
 'Pictures',
 '.zprofile',
 'datasets',
 'submission.csv',
 'ML Assignment2.ipynb',
 'Postman',
 '2024dc04116_Sheetal Bohra',
 '.nuget',
 '.zsh_history',
 '.ipython',
 'Desktop',
 'Library',
 '.matplotlib',
 'models',
 '.lesshst',
 '.cups',
 'logistic_regression.joblib',
 'logistic_regression_metrics.csv',
 '.codex',
 'Public',
 'ml',
 '.idlerc',
 '.tcshrc',
 'ds-env',
 'Movies',
 'Applications',
 '.Trash',
 '.ipynb_checkpoints',
 '.jupyter',
 'Documents',
 'venv',
 '.vscode',
 '.bash_profile',
 'Downloads',
 'decision_tree.joblib',
 '.zsh_sessions',
 '2024dc04116_Sheetal Bohra.ipynb',
 '.conda']

In [32]:
######################### 3. KNN Classifier ######################

In [33]:
import os
os.makedirs("models", exist_ok=True)
os.makedirs("metrics", exist_ok=True)

In [34]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)


In [35]:
# df should already be loaded and cleaned (drop " ?" and NaNs)
X = df.drop("income", axis=1)
y = df["income"].map({"<=50K": 0, ">50K": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [87]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor_knn = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


In [88]:

knn_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_knn),
        ("classifier", KNeighborsClassifier(
            n_neighbors=21,
            weights="distance",
            algorithm="brute",
            metric="euclidean",
            n_jobs=-1
        ))
    ]
)


In [89]:
knn_model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [90]:
y_pred_knn = knn_model.predict(X_test)
y_proba_knn = knn_model.predict_proba(X_test)[:, 1]


In [91]:
accuracy_knn = accuracy_score(y_test, y_pred_knn)
auc_knn = roc_auc_score(y_test, y_proba_knn)
precision_knn = precision_score(y_test, y_pred_knn)
recall_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)
mcc_knn = matthews_corrcoef(y_test, y_pred_knn)

print("KNN Results")
print("Accuracy:", accuracy_knn)
print("AUC:", auc_knn)
print("Precision:", precision_knn)
print("Recall:", recall_knn)
print("F1:", f1_knn)
print("MCC:", mcc_knn)


KNN Results
Accuracy: 0.8406264394288346
AUC: 0.8915027676894822
Precision: 0.6898280802292264
Recall: 0.6141581632653061
F1: 0.6497975708502024
MCC: 0.5486336793235472


In [92]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_knn))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_knn))



Confusion Matrix:
[[4512  433]
 [ 605  963]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.91      0.90      4945
           1       0.69      0.61      0.65      1568

    accuracy                           0.84      6513
   macro avg       0.79      0.76      0.77      6513
weighted avg       0.84      0.84      0.84      6513



In [94]:
joblib.dump(knn_model, "models/knn.joblib")
print("KNN model saved successfully")

metrics_knn = pd.DataFrame({
    "Model": ["KNN"],
    "Accuracy": [accuracy_knn],
    "AUC": [auc_knn],
    "Precision": [precision_knn],
    "Recall": [recall_knn],
    "F1": [f1_knn],
    "MCC": [mcc_knn]
})

metrics_knn.to_csv("metrics/knn_metrics.csv", index=False)
metrics_knn


KNN model saved successfully


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,KNN,0.840626,0.891503,0.689828,0.614158,0.649798,0.548634


In [43]:
################################### 4.Naive Bayes #######################################

In [44]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)


In [45]:
X = df.drop("income", axis=1)
y = df["income"].map({"<=50K": 0, ">50K": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [46]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor_nb = ColumnTransformer(
    transformers=[
        ("num", MinMaxScaler(), numerical_cols),  # ensures non-negative numeric features
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


In [47]:
nb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_nb),
        ("classifier", MultinomialNB(alpha=1.0))
    ]
)


In [48]:
nb_model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [49]:
y_pred_nb = nb_model.predict(X_test)
y_proba_nb = nb_model.predict_proba(X_test)[:, 1]


In [50]:
accuracy_nb = accuracy_score(y_test, y_pred_nb)
auc_nb = roc_auc_score(y_test, y_proba_nb)
precision_nb = precision_score(y_test, y_pred_nb)
recall_nb = recall_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)
mcc_nb = matthews_corrcoef(y_test, y_pred_nb)

print("Naive Bayes (MultinomialNB) Results")
print("Accuracy:", accuracy_nb)
print("AUC:", auc_nb)
print("Precision:", precision_nb)
print("Recall:", recall_nb)
print("F1:", f1_nb)
print("MCC:", mcc_nb)


Naive Bayes (MultinomialNB) Results
Accuracy: 0.7944111776447106
AUC: 0.8730747405129899
Precision: 0.5553942912433478
Recall: 0.7321428571428571
F1: 0.631636863823934
MCC: 0.5018007121591272


In [51]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))



Confusion Matrix:
[[4026  919]
 [ 420 1148]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.81      0.86      4945
           1       0.56      0.73      0.63      1568

    accuracy                           0.79      6513
   macro avg       0.73      0.77      0.74      6513
weighted avg       0.82      0.79      0.80      6513



In [52]:
joblib.dump(nb_model, "models/naive_bayes.joblib")
print("Naive Bayes model saved successfully")

metrics_nb = pd.DataFrame({
    "Model": ["Naive Bayes"],
    "Accuracy": [accuracy_nb],
    "AUC": [auc_nb],
    "Precision": [precision_nb],
    "Recall": [recall_nb],
    "F1": [f1_nb],
    "MCC": [mcc_nb]
})

metrics_nb.to_csv("metrics/naive_bayes_metrics.csv", index=False)
metrics_nb


Naive Bayes model saved successfully


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Naive Bayes,0.794411,0.873075,0.555394,0.732143,0.631637,0.501801


In [53]:
########################## 5.Random Forest #################################

In [54]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)


In [55]:
X = df.drop("income", axis=1)
y = df["income"].map({"<=50K": 0, ">50K": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [80]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


In [81]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_rf),
        ("classifier", RandomForestClassifier(
            n_estimators=400,
            random_state=42,
            n_jobs=-1,
            max_depth=20,
            min_samples_split=20,
            min_samples_leaf=10,
            max_features="sqrt"
        ))
    ]
)



In [82]:
rf_model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [83]:
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]


In [84]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
mcc_rf = matthews_corrcoef(y_test, y_pred_rf)

print("Random Forest Results")
print("Accuracy:", accuracy_rf)
print("AUC:", auc_rf)
print("Precision:", precision_rf)
print("Recall:", recall_rf)
print("F1:", f1_rf)
print("MCC:", mcc_rf)


Random Forest Results
Accuracy: 0.857362198679564
AUC: 0.9109597150285796
Precision: 0.7723785166240409
Recall: 0.5778061224489796
F1: 0.6610726012404232
MCC: 0.5827900693726188


In [85]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))



Confusion Matrix:
[[4678  267]
 [ 662  906]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91      4945
           1       0.77      0.58      0.66      1568

    accuracy                           0.86      6513
   macro avg       0.82      0.76      0.79      6513
weighted avg       0.85      0.86      0.85      6513



In [86]:
joblib.dump(rf_model, "models/random_forest.joblib")
print("Random Forest model saved successfully")

metrics_rf = pd.DataFrame({
    "Model": ["Random Forest"],
    "Accuracy": [accuracy_rf],
    "AUC": [auc_rf],
    "Precision": [precision_rf],
    "Recall": [recall_rf],
    "F1": [f1_rf],
    "MCC": [mcc_rf]
})

metrics_rf.to_csv("metrics/random_forest_metrics.csv", index=False)
metrics_rf


Random Forest model saved successfully


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Random Forest,0.857362,0.91096,0.772379,0.577806,0.661073,0.58279


In [63]:
############################################### 6.XGBoost ##################################################

In [64]:
import sys
!{sys.executable} -m pip uninstall -y xgboost
!{sys.executable} -m pip install "xgboost==2.0.3"


Found existing installation: xgboost 2.0.3
Uninstalling xgboost-2.0.3:
  Successfully uninstalled xgboost-2.0.3
Defaulting to user installation because normal site-packages is not writeable
  Using cached xgboost-2.0.3-py3-none-macosx_12_0_arm64.whl.metadata (2.0 kB)
Using cached xgboost-2.0.3-py3-none-macosx_12_0_arm64.whl (1.9 MB)


In [65]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

from xgboost import XGBClassifier


In [66]:
X = df.drop("income", axis=1)
y = df["income"].map({"<=50K": 0, ">50K": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [67]:
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(exclude=["object"]).columns

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)


In [68]:
xgb_clf = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1,
    gamma=0,
    objective="binary:logistic",
    eval_metric="auc",
    n_jobs=-1,
    random_state=42
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_xgb),
        ("classifier", xgb_clf)
    ]
)


In [69]:
xgb_model.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [70]:
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]


In [71]:
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
auc_xgb = roc_auc_score(y_test, y_proba_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb)
recall_xgb = recall_score(y_test, y_pred_xgb)
f1_xgb = f1_score(y_test, y_pred_xgb)
mcc_xgb = matthews_corrcoef(y_test, y_pred_xgb)

print("XGBoost Results")
print("Accuracy:", accuracy_xgb)
print("AUC:", auc_xgb)
print("Precision:", precision_xgb)
print("Recall:", recall_xgb)
print("F1:", f1_xgb)
print("MCC:", mcc_xgb)


XGBoost Results
Accuracy: 0.8697988638108398
AUC: 0.9251073027795547
Precision: 0.7727272727272727
Recall: 0.6505102040816326
F1: 0.7063711911357341
MCC: 0.6273304660423282


In [72]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))



Confusion Matrix:
[[4645  300]
 [ 548 1020]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.94      0.92      4945
           1       0.77      0.65      0.71      1568

    accuracy                           0.87      6513
   macro avg       0.83      0.79      0.81      6513
weighted avg       0.87      0.87      0.87      6513



In [73]:
joblib.dump(xgb_model, "models/xgboost.joblib")
print("XGBoost model saved successfully")

metrics_xgb = pd.DataFrame({
    "Model": ["XGBoost"],
    "Accuracy": [accuracy_xgb],
    "AUC": [auc_xgb],
    "Precision": [precision_xgb],
    "Recall": [recall_xgb],
    "F1": [f1_xgb],
    "MCC": [mcc_xgb]
})

metrics_xgb.to_csv("metrics/xgboost_metrics.csv", index=False)
metrics_xgb


XGBoost model saved successfully


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,XGBoost,0.869799,0.925107,0.772727,0.65051,0.706371,0.62733


In [95]:
joblib.dump(knn_model, "knn.joblib")
print("KNN saved")
joblib.dump(nb_model, "naive_bayes.joblib")
print("Naive Bayes saved")
joblib.dump(rf_model, "random_forest.joblib")
print("Random Forest saved")
joblib.dump(xgb_model, "xgboost.joblib")
print("XGBoost saved")


KNN saved
Naive Bayes saved
Random Forest saved
XGBoost saved


In [97]:
metrics_knn.to_csv("knn_metrics.csv", index=False)
metrics_nb.to_csv("naive_bayes_metrics.csv", index=False)
metrics_rf.to_csv("random_forest_metrics.csv", index=False)
metrics_xgb.to_csv("xgboost_metrics.csv", index=False)


In [98]:
os.listdir()


['knn.joblib',
 '.zshrc.save',
 '.config',
 'metrics',
 'Music',
 '.zprofile.pysave',
 'App.ipynb',
 '.DS_Store',
 'Housing.ipynb',
 '.CFUserTextEncoding',
 'decision_tree_metrics.csv',
 '.xonshrc',
 'knn_metrics.csv',
 'Untitled.ipynb',
 '.zshrc',
 '.local',
 'Pictures',
 '.zprofile',
 'datasets',
 'submission.csv',
 'ML Assignment2.ipynb',
 'Postman',
 '2024dc04116_Sheetal Bohra',
 '.nuget',
 'naive_bayes_metrics.csv',
 '.zsh_history',
 '.ipython',
 'Desktop',
 'Library',
 '.matplotlib',
 'models',
 '.lesshst',
 'random_forest_metrics.csv',
 '.cups',
 'logistic_regression.joblib',
 'logistic_regression_metrics.csv',
 'random_forest.joblib',
 '.codex',
 'Public',
 'naive_bayes.joblib',
 'ml',
 '.idlerc',
 'xgboost_metrics.csv',
 '.tcshrc',
 'ds-env',
 'xgboost.joblib',
 'Movies',
 'Applications',
 '.Trash',
 '.ipynb_checkpoints',
 '.jupyter',
 'Documents',
 'venv',
 '.vscode',
 '.bash_profile',
 'Downloads',
 'decision_tree.joblib',
 '.zsh_sessions',
 '2024dc04116_Sheetal Bohra.ipynb'

In [99]:
import os
os.makedirs("model", exist_ok=True)
os.makedirs("metrics", exist_ok=True)


In [100]:
import shutil

# move models
shutil.move("logistic_regression.joblib", "model/logistic_regression.joblib")
shutil.move("decision_tree.joblib", "model/decision_tree.joblib")
shutil.move("knn.joblib", "model/knn.joblib")
shutil.move("naive_bayes.joblib", "model/naive_bayes.joblib")
shutil.move("random_forest.joblib", "model/random_forest.joblib")
shutil.move("xgboost.joblib", "model/xgboost.joblib")

# move metrics
shutil.move("logistic_regression_metrics.csv", "metrics/logistic_regression_metrics.csv")
shutil.move("decision_tree_metrics.csv", "metrics/decision_tree_metrics.csv")
shutil.move("knn_metrics.csv", "metrics/knn_metrics.csv")
shutil.move("naive_bayes_metrics.csv", "metrics/naive_bayes_metrics.csv")
shutil.move("random_forest_metrics.csv", "metrics/random_forest_metrics.csv")
shutil.move("xgboost_metrics.csv", "metrics/xgboost_metrics.csv")


'metrics/xgboost_metrics.csv'